# T27 / E08 — Bỏ bằng chứng đi rồi đo lại chú ý

**Phép kiểm can thiệp duy nhất của cả đề tài.**

Mọi thí nghiệm đến giờ đều là **tương quan**: đo một tín hiệu, so với nhãn người khác gán, báo
mức khớp. Khi hai bên khớp nhau thì kết luận "cơ chế hoạt động" là một **suy luận**, không phải
một quan sát. E06 gần trực tiếp nhất, nhưng nó vẫn chỉ *quan sát* chú ý rơi ở đâu.

Cái này **can thiệp**. Mỗi claim được ghép cặp với chính nó: cùng phản hồi, đọc hai lần trên hai
ngữ cảnh mười câu **khác nhau đúng một câu ở đúng một vị trí**. Một bên có câu bằng chứng vàng,
bên kia thay bằng một câu nhiễu. Không gì khác thay đổi — không độ dài, không số đoạn, không thứ
tự, không nhãn nào do người gán.

Nếu tín hiệu chunk-aware đúng như đề tài nói, bỏ bằng chứng đi phải làm phân bố chú ý **tản ra**:
không còn gì trong ngữ cảnh để tập trung vào. Nếu nó không nhúc nhích, thì thứ đặc trưng hình
dạng bắt được trên các bộ khác **không phải** "mô hình đã tìm thấy bằng chứng".

## Nhãn nửa 'absent' không do ai gán

Rút câu vàng ra thì phản hồi khẳng định điều ngữ cảnh không chứa — đó là định nghĩa của ảo giác
ngoại lai. Nhãn `extrinsic` ở nửa ấy là **sự thật về cách dựng**, không phải một phán đoán ai đó
có thể bất đồng. Đây là điều làm E08 khác mọi thí nghiệm còn lại: T13 đã đo rằng ranh giới nội
tại–ngoại lai đánh bại hai người gán nhãn và cả Gemini, và ở đây ranh giới ấy được **dựng ra**
chứ không được **đoán**.

## Vì sao là ViWikiFC

Bộ duy nhất mà nhãn NEI **có bằng chứng** (100 % nguyên văn, xác nhận ở T11), và bộ duy nhất đủ
nhỏ — 3.814 câu từ 73 bài Wikipedia — để làm kho truy xuất cho chính nó. Mục 8 `docs/DATA.md` dự
liệu việc này từ đầu, và T16 đã dựng kho.

## Đã đo trên CPU trước khi đặt lịch GPU

```
  bằng chứng vàng có trong kho    2.090/2.090  (100 %)
  ngữ cảnh top-10                 496 token TB, 9,8 đoạn sau khi chia theo câu
  hai vế CÙNG số đoạn             87,2 %
  và khác ĐÚNG một đoạn          99,7 % trong số đó  → chỉ giữ những cặp này
  trong đó khác đúng một đoạn     100 %
  dựng được                       1.836 cặp = 3.672 dòng
```

**Một confound suýt lọt.** BM25 để câu vàng ở **hạng 0 với 94 % claim** — vị trí tương đối trung
bình 0,040. Nếu ghép theo thứ tự truy xuất thì một bộ đoán *"luôn chọn đoạn đầu"* đã thắng sàn,
và hit@1 cao sẽ chẳng chứng minh được gì. **Xáo thứ tự** mười câu trước khi ghép đưa vị trí vàng
về **0,513**, rải đều mười hạng. Cả hai vế của một cặp dùng **cùng một hoán vị**, nếu không thì
chúng khác nhau ở mười chỗ chứ không phải một.

## Hướng dự đoán, ghi trước khi thấy số

| đặc trưng | bỏ vàng đi thì phải | vì sao |
|---|---|---|
| `chunk_entropy` | **tăng** | không còn gì để tập trung |
| `chunk_max_share` | **giảm** | không đoạn nào còn trội hẳn |
| `chunk_gini` | **giảm** | các đoạn đều nhau hơn |
| `top1_top2_gap` | **giảm** | đoạn tốt nhất hết nổi bật |
| `chunk_drift` | *không dự đoán* | drift nói về chuyển động theo token |

Bảng này nằm trong `run_extrinsic.py` dưới dạng `EXPECTED_DIRECTION`, có ca kiểm thử khóa lại, để
một kết quả đi ngược **không thể** được mô tả lại thành xác nhận sau khi đã thấy số.

**Kết quả đi ngược là kết quả bác cách diễn giải của đề tài**, và phải báo cáo đúng như vậy.

## Chuẩn bị

Ô 1 giống notebook T26. Ô 2 **khác**: E08 cần `rank-bm25` cho chỉ mục BM25, mà `pip install --no-deps` cố ý không kéo phụ thuộc nào về.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 10c9e95 T26: công cụ E07 chunk-aware trên ngữ cảnh dài, chờ chạy GPU (#62)


In [ ]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit của mô hình đọc 7B.
#
# rank-bm25 là thứ T27 phải thêm. Nó có trong pyproject.toml, nhưng `pip install --no-deps -e .`
# cố ý không kéo phụ thuộc nào về, nên pip vẫn in "requires rank-bm25, which is not installed"
# ở MỌI lượt chạy từ T23. Cảnh báo ấy vô hại cho T23-T26 và KHÔNG vô hại cho T27: E08 dựng chỉ
# mục BM25 nên hỏng ngay ở giây thứ 43.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes rank-bm25

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
#
# Bài học T26: hai ô chấm điểm chạy sau 10 giờ GPU rồi mới báo thiếu shard của phiên trước. Ô kiểm
# toàn vẹn có báo đúng, nhưng nó đứng SAU nên vô dụng. Ô này đứng đầu và dừng ngay.
#
# Quy tắc đã chốt ở T26: notebook trên Kaggle CHỈ làm việc trích. Mọi shard mà các ô sau cần đều
# phải do chính phiên này sinh ra — phiên Kaggle mới không có gì khác.
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

cfg = load_config("configs/e08_extrinsic_viwikifc.yaml")
run = extraction_hash(cfg)
will_extract = {("viwikifc_e08", "dev")}          # ô 6 sinh ra đúng cái này
needed = {(cfg.dataset.name, "dev")}              # ô 7 cần đúng cái này

missing = needed - will_extract
print(f"  hash trích        : {run}")
print(f"  phiên này sẽ sinh : {sorted(will_extract)}")
print(f"  các ô sau cần     : {sorted(needed)}")
for dataset, split in sorted(needed):
    path = Path("data/processed") / f"{dataset}_{split}_{run}.jsonl"
    have = path.exists()
    print(f"  {'đã có' if have else 'sẽ trích'}: {path.name}")
if missing:
    raise SystemExit(f"THIEU {sorted(missing)} - phien Kaggle moi khong co shard cua phien truoc.")

# Goi ma CAC O SAU that su import. Khai ro thay vi quet toan bo pyproject: fastapi, uvicorn va
# ruff cung duoc khai bao nhung Kaggle khong cai va E08 khong dung, nen quet tat ca se bao dong
# gia - dung loi ma o kiem toan ven cua T26 tung mac.
#
# rank_bm25 la ly do o nay ton tai. Pip in "requires rank-bm25, which is not installed" o MOI
# luot chay tu T23; canh bao ay vo hai cho T23-T26 va khong vo hai cho T27.
absent = [name for name in ("rank_bm25", "torch", "transformers", "bitsandbytes", "pandas")
          if importlib.util.find_spec(name) is None]
print(f"  goi cac o sau can : {'thieu ' + str(absent) if absent else 'du ca'}")
if absent:
    raise SystemExit(f"THIEU goi {absent} - them vao o cai dat roi chay lai.")
print("\nTien kiem dat: shard va phu thuoc deu san sang.")

In [ ]:
# Ô 4 — dựng ngữ cảnh truy xuất theo cặp. Khoảng 2 phút, CPU.
# Không cần GPU: chỉ là BM25 trên 3.814 câu rồi ghép văn bản.
!python scripts/build_evidence_corpus.py
!python scripts/build_retrieval_contexts.py --split dev

In [ ]:
# Ô 5 — kiểm tiền đề của thí nghiệm trên chính dữ liệu vừa dựng. Khoảng 1 phút, CPU.
# Ba điều phải đúng, nếu không thì phép so cặp mất nghĩa và đừng tốn GPU chạy tiếp.
import sys

sys.path.insert(0, "src")
import numpy as np

from vihallulens.data.chunking import chunk_context
from vihallulens.data.loading import load_dataset

d = load_dataset("viwikifc_e08", "dev")
present = d[d["meta"].str["condition"] == "present"].sort_values("sample_id")
absent = d[d["meta"].str["condition"] == "absent"].sort_values("sample_id")
print(f"cặp                : {len(present):,} present / {len(absent):,} absent")
assert len(present) == len(absent), "cặp lệch nửa"

def cut(text):
    return [c.text for c in chunk_context(text, strategy="sentence", min_words=5)]

same, one = 0, 0
for a, b in zip(present["context"], absent["context"], strict=True):
    left, right = cut(a), cut(b)
    if len(left) == len(right):
        same += 1
        one += int(sum(x != y for x, y in zip(left, right, strict=True)) == 1)
print(f"hai vế cùng số đoạn: {same:,}/{len(present):,} ({same / len(present):.1%})")
print(f"khác đúng một đoạn : {one:,}/{same:,} ({one / same:.1%})")

pos = np.asarray([m["gold_position"] for m in present["meta"]])
k = np.median([len(cut(t)) for t in present["context"].head(200)])
print(f"vị trí đoạn vàng   : TB {pos.mean() / (k - 1):.3f}, ở đoạn đầu {(pos == 0).mean():.1%}")

assert same == len(present), "co cap lech so doan lot qua khau dung"
assert one == same, "co cap khac nhieu hon mot doan"
assert 0.35 < pos.mean() / (k - 1) < 0.65, "vi tri vang lech ve mot phia"
print("\nBa tien de deu dat.")

## Trích đặc trưng

Một ô, khoảng **48 phút**: 3.672 dòng ở ~784 ms mỗi dòng. Chạy lại được.

**Đọc gì trong lúc chạy:** dòng `mẫu có bằng chứng` phải báo khoảng 1.836/3.672 — đúng một nửa,
vì chỉ nửa `present` mang bằng chứng. Nếu nó báo 0 thì cột `evidence` không tới được bộ trích và
phần định vị sẽ trống.

In [ ]:
# Ô 6 — trích đặc trưng. Khoảng 48 phút.
!python scripts/extract_features.py --config configs/e08_extrinsic_viwikifc.yaml --split dev

## Đo

Chạy CPU, vài giây.

In [ ]:
# Ô 7 — phép kiểm can thiệp và phần định vị.
!python scripts/run_extrinsic.py --config configs/e08_extrinsic_viwikifc.yaml --split dev

In [ ]:
# Ô 8 — kiểm toàn vẹn shard trước khi rời phiên. Vài giây, CPU.
import json
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

run = extraction_hash(load_config("configs/e08_extrinsic_viwikifc.yaml"))
path = Path("data/processed") / f"viwikifc_e08_dev_{run}.jsonl"
if not path.exists():
    raise SystemExit(f"THIEU {path.name} - chay lai o trich truoc khi roi phien.")
rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
ids = {r["sample_id"] for r in rows}
blocks = [k for k in rows[0] if k.startswith(("lookback_", "chunk_", "top1_"))]
gold = [r for r in rows if "gold_rank" in r]
ok = len(rows) == 3672 and len(ids) == len(rows) and len(blocks) == 7
print(f"  {path.name}")
print(f"  {len(rows):,}/3.672 dong, {len(ids):,} id, {len(blocks)} khoi dac trung")
print(f"  {len(gold):,} mau co gold_rank (nua 'present')")
print(f"  bi cat ngu canh: {sum(r['truncated'] for r in rows):,}")
print("\nShard hop le." if ok else "\nCO VAN DE - chay lai o trich truoc khi roi phien.")

In [ ]:
# Ô 9 — lấy kết quả về.
import shutil
from pathlib import Path

for name in ("results/runs.jsonl", "data/interim/viwikifc_e08_dev.parquet"):
    shutil.copy(name, f"/kaggle/working/{Path(name).name}")
    print(Path(name).name)
for path in sorted(Path("data/processed").glob("viwikifc_e08_*.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:,.0f} MB")

## Sau khi chạy

Dán toàn bộ output của ô 7. Bảng cần lấy là `BỎ BẰNG CHỨNG ĐI THÌ CHÚ Ý CÓ ĐỔI KHÔNG` với đủ cả
ba cột `đổi`, `% cặp đúng hướng` và `cỡ ảnh hưởng`, cộng khối `ĐỊNH VỊ TRÊN NỬA CÓ VÀNG`.

**Đọc cột `đổi` cùng lúc với cột phần trăm.** Kiểm định theo cặp đo mức nhất quán của *hướng*,
không đo độ lớn — một dịch chuyển nhỏ tới mức vô nghĩa vẫn cho 100 % cặp đúng hướng nếu nó đều.
Có ca kiểm thử dựng sẵn đúng cái bẫy ấy.

Rồi Quick Save, tải notebook về, chép đè lên `notebooks/t27_bo_bang_chung_t4.ipynb`. **Đừng dùng
Save & Run All.**

Tải cả `runs.jsonl`, `viwikifc_e08_dev.parquet` và shard `viwikifc_e08_dev_*.jsonl` về — chấm
điểm sẽ chạy lại ở máy cá nhân theo quy tắc chốt ở T26, để mọi con số đem so đều cùng một máy.